<a href="https://colab.research.google.com/github/pmcray/cosmic_engine/blob/main/fluid_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import taichi as ti
import numpy as np
from IPython.display import Video, display

# Initialize Taichi to use the GPU (Colab's T4 or V100)
ti.init(arch=ti.gpu)

RES = 512
dt = 0.03

velocity = ti.Vector.field(2, dtype=float, shape=(RES, RES))
new_velocity = ti.Vector.field(2, dtype=float, shape=(RES, RES))
dye = ti.Vector.field(3, dtype=float, shape=(RES, RES))
new_dye = ti.Vector.field(3, dtype=float, shape=(RES, RES))

@ti.func
def bilinear_interp(field, p):
    p = ti.max(0.0, ti.min(float(RES - 1.001), p))
    i, j = int(p[0]), int(p[1])
    fx = p[0] - i
    fy = p[1] - j

    c00 = field[i, j]
    c10 = field[i + 1, j]
    c01 = field[i, j + 1]
    c11 = field[i + 1, j + 1]

    return (c00 * (1.0 - fx) + c10 * fx) * (1.0 - fy) + \
           (c01 * (1.0 - fx) + c11 * fx) * fy

@ti.kernel
def advect(field: ti.template(), new_field: ti.template(), vec_field: ti.template()):
    for i, j in field:
        p = ti.Vector([float(i), float(j)])
        back_p = p - vec_field[i, j] * dt
        new_field[i, j] = bilinear_interp(field, back_p)
        new_field[i, j] *= 0.995  # Slight fade

@ti.kernel
def inject_zonal_jets():
    for i, j in velocity:
        # Create horizontal bands of wind
        if j > RES * 0.45 and j < RES * 0.55:
            wind_speed = ti.sin((float(j) / RES) * 30.0) * 150.0
            velocity[i, j][0] = wind_speed

            # Inject ammonia cloud colors
            if ti.random() > 0.98:
                dye[i, j] = ti.Vector([0.8, 0.7, 0.6])

@ti.kernel
def copy_fields():
    for i, j in velocity:
        velocity[i, j] = new_velocity[i, j]
        dye[i, j] = new_dye[i, j]

def main():
    print("Initializing Cosmic Engine renderer...")

    # Use Taichi's VideoManager to silently write frames to disk
    video_manager = ti.tools.VideoManager(output_dir="./frames", framerate=30, automatic_build=False)

    total_frames = 200
    for frame in range(total_frames):
        inject_zonal_jets()
        advect(velocity, new_velocity, velocity)
        advect(dye, new_dye, velocity)
        copy_fields()

        # Grab the frame data, ensure it's clamped to valid RGB values, and save it
        img = np.clip(dye.to_numpy(), 0.0, 1.0)
        video_manager.write_frame(img)

        if frame % 50 == 0:
            print(f"Rendered frame {frame}/{total_frames}")

    print("Stitching frames into MP4...")
    video_manager.make_video(gif=False, mp4=True)
    print("Done!")

# Run the simulation
main()

# Display the output directly in the Colab cell
display(Video('./frames/video.mp4', embed=True, width=512))